# Why self-attention is called "self"
One attention function, used two ways: between two sequences (Luong's dot attention in an encoder-decoder) and within one sequence (self-attention). The networks are untrained: only the computation is compared, not the meaning of the weights.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""       # no training: the CPU is enough
import numpy as np
import keras
np.set_printoptions(precision=3, suppress=True)
keras.utils.set_random_seed(0)

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def attention(queries, keys, values):
    """The three equations: scores by dot product, softmax, weighted sum of the values."""
    A = softmax(queries @ keys.T)
    return A, A @ values

## A real sentence pair from the English-French dataset

In [2]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf-8").read().split("\n")
en, fr = next(l.split("\t")[:2] for l in lines if l.startswith("Turn off the light.\t"))
src = en.lower().strip(".").split()
tgt = fr.lower().strip(".").split()
print(src, tgt)

['turn', 'off', 'the', 'light'] ['éteins', 'la', 'lumière']


## 1. Between two sequences: Luong's dot attention
An (untrained) LSTM encoder reads the English words and gives one hidden state h_j per word; an LSTM decoder, started from the encoder's final state, gives one hidden state s_i per French word. Embeddings start as N(0, 1) numbers so that the weights are not all equal.

In [3]:
d = 8
emb_src = keras.layers.Embedding(len(src), d, embeddings_initializer=keras.initializers.RandomNormal(stddev=1.0, seed=1))
emb_tgt = keras.layers.Embedding(len(tgt), d, embeddings_initializer=keras.initializers.RandomNormal(stddev=1.0, seed=2))
encoder = keras.layers.LSTM(d, return_sequences=True, return_state=True)
decoder = keras.layers.LSTM(d, return_sequences=True)
X_src = emb_src(np.arange(len(src))[None])                # 1 x 4 x 8 embeddings
H, h_last, c_last = encoder(X_src)
S = decoder(emb_tgt(np.arange(len(tgt))[None]), initial_state=[h_last, c_last])
H, S = H.numpy()[0], S.numpy()[0]
print("encoder states H:", H.shape, "  decoder states S:", S.shape)

A_luong, C = attention(S, H, H)                        # query = s_i, key = h_j, value = h_j
print("weights (French word x English word):\n", A_luong)
A_k = keras.layers.Attention()([S[None], H[None]], return_attention_scores=True)
assert np.allclose(A_luong, A_k[1].numpy()[0], atol=1e-5) and np.allclose(C, A_k[0].numpy()[0], atol=1e-5)
print("same as keras.layers.Attention (Luong-style): True")

encoder states H: (4, 8)   decoder states S: (3, 8)
weights (French word x English word):
 [[0.224 0.262 0.253 0.26 ]
 [0.232 0.258 0.262 0.249]
 [0.249 0.245 0.259 0.248]]
same as keras.layers.Attention (Luong-style): True


## 2. Within one sequence: self-attention
The same function, but queries, keys and values all come from the English sentence. First without projections, then with W_Q, W_K, W_V and the 1/sqrt(d_k) scaling of the transformer.

In [4]:
X = X_src.numpy()[0]
A_self_plain, _ = attention(X, X, X)
print("plain self-attention weights (English x English):\n", A_self_plain)

Wq, Wk, Wv = np.random.default_rng(0).normal(0, 1 / np.sqrt(d), (3, d, d))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
A_self, Y = attention(Q / np.sqrt(d), K, V)           # dividing the queries divides every score
print("transformer self-attention weights:\n", A_self)
A_k = keras.layers.Attention()([Q[None] / np.sqrt(d), V[None], K[None]], return_attention_scores=True)
assert np.allclose(A_self, A_k[1].numpy()[0], atol=1e-5) and np.allclose(Y, A_k[0].numpy()[0], atol=1e-5)
print("same Keras layer, same result: True")

plain self-attention weights (English x English):
 [[0.774 0.02  0.131 0.076]
 [0.    1.    0.    0.   ]
 [0.    0.    1.    0.   ]
 [0.127 0.139 0.009 0.725]]
transformer self-attention weights:
 [[0.192 0.243 0.352 0.213]
 [0.302 0.012 0.326 0.36 ]
 [0.134 0.181 0.432 0.252]
 [0.257 0.259 0.286 0.198]]
same Keras layer, same result: True


## 3. One layer, two uses
Keras' `MultiHeadAttention` does not know whether it is doing self-attention. Called with query = value = the English sentence it is self-attention (4 x 4 weights); called with the French states as query it attends from one sequence to the other (3 x 4 weights).

In [5]:
mha = keras.layers.MultiHeadAttention(num_heads=1, key_dim=d)
_, w_self = mha(X_src, X_src, return_attention_scores=True)
_, w_cross = mha(S[None], X_src, return_attention_scores=True)
print("query = value = English:       weights", tuple(w_self.shape[2:]))
print("query = French, value = English: weights", tuple(w_cross.shape[2:]))

query = value = English:       weights (4, 4)
query = French, value = English: weights (3, 4)


## Saving the three weight matrices for the figure

In [6]:
import pandas as pd
pd.DataFrame(A_luong, index=tgt, columns=src).to_csv("data/weights_luong.csv", float_format="%.4f")
pd.DataFrame(A_self_plain, index=src, columns=src).to_csv("data/weights_self_plain.csv", float_format="%.4f")
pd.DataFrame(A_self, index=src, columns=src).to_csv("data/weights_self.csv", float_format="%.4f")